# 18 · Files and Streams

*Beyond the contest* · 6 exercises

Read and write real files: numbers and words line by line, reports and CSV records, and streams that handle a file of any size without loading it whole.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-18) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-18)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import csv
import re
import tempfile
from pathlib import Path

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 18.1 · Sum the numbers in a file  `Core`

**Task.** Write `sum_file(path)` that returns the sum of the integers in a text file, one per line, skipping blank lines. Open it with `with open(path) as f` and iterate over `f`, which yields one line at a time.

**Example.** a file holding `3\n\n-1\n10\n` -> `12`

**Big O.** O(size of the file) time, O(1) extra space when read line by line.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-1)

In [ ]:
# 18.1 Sum the numbers in a file
def sum_file(path):
    raise NotImplementedError

In [ ]:
def test_ex_18_1():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / 'n.txt').write_text('3\n\n-1\n10\n')
        (tmp / 'empty.txt').write_text('')
        assert sum_file(tmp / 'n.txt') == 12
        assert sum_file(tmp / 'empty.txt') == 0


test_ex_18_1()
print('18.1 passed')

### 18.2 · Most common words in a file  `Core`

**Task.** A word is a run of letters, compared in lower case. Write `top_words(path, k)` that returns the `k` most common words in a file as `(word, count)` pairs, most frequent first and ties in alphabetical order. Count with a dictionary (or `collections.Counter`), then sort by `(-count, word)`.

**Example.** `The cat and the hat.\nThe end, and THE start!\n`, `2` -> `[('the', 4), ('and', 2)]`

**Big O.** O(size + W log W) for W distinct words: one pass to count, one sort.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-2)

In [ ]:
# 18.2 Most common words in a file
def top_words(path, k):
    raise NotImplementedError

In [ ]:
def test_ex_18_2():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / 'w.txt').write_text('The cat and the hat.\nThe end, and THE start!\n')
        assert top_words(tmp / 'w.txt', 2) == [('the', 4), ('and', 2)]
        assert top_words(tmp / 'w.txt', 4) == [('the', 4), ('and', 2), ('cat', 1), ('end', 1)]
        assert len(top_words(tmp / 'w.txt', 99)) == 6


test_ex_18_2()
print('18.2 passed')

### 18.3 · Write a report  `Core`

**Task.** Write `write_report(path, scores)` that writes one line `name,score` per entry of the dictionary `scores`, highest score first and ties by name, replacing any existing file, and returns how many lines it wrote. Open with mode `"w"` and write each line with `\n`.

**Example.** `{'ana': 7, 'bo': 9, 'cy': 7}` -> writes `bo,9\nana,7\ncy,7\n` and returns `3`

**Big O.** O(N log N) for the sort, O(N) to write.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-3)

In [ ]:
# 18.3 Write a report
def write_report(path, scores):
    raise NotImplementedError

In [ ]:
def test_ex_18_3():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        path = tmp / 'report.csv'
        path.write_text('old contents\n')
        assert write_report(path, {'ana': 7, 'bo': 9, 'cy': 7}) == 3
        assert path.read_text() == 'bo,9\nana,7\ncy,7\n'
        assert write_report(path, {}) == 0
        assert path.read_text() == ''


test_ex_18_3()
print('18.3 passed')

### 18.4 · Stream a large file  `Practice`

**Task.** Write `longest_line(path)` that returns the length of the longest line in a file (not counting the newline), holding only one line in memory at a time: iterate over the open file, never call `read()` or `readlines()`.

**Example.** a file with lines `ab`, `abcd`, `` -> `4`

**Big O.** O(size) time and O(longest line) memory: only one line is ever held, however big the file is.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-4)

In [ ]:
# 18.4 Stream a large file
def longest_line(path):
    raise NotImplementedError

In [ ]:
def test_ex_18_4():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / 'a.txt').write_text('ab\nabcd\n\n')
        with open(tmp / 'big.txt', 'w') as f:
            for i in range(100000):
                f.write('x' * (i % 97) + '\n')
            f.write('y' * 500)
        assert longest_line(tmp / 'a.txt') == 4
        assert longest_line(tmp / 'big.txt') == 500


test_ex_18_4()
print('18.4 passed')

### 18.5 · Copy matching lines to another file  `Practice`

**Task.** Write `copy_matching(source, target, needle)` that writes every line of `source` containing `needle` to `target` (in order, newlines kept) and returns how many it wrote. Open both files in one `with` statement and stream from one to the other.

**Example.** lines `error: disk`, `ok`, `error: net` with needle `error` -> writes the two error lines, returns `2`

**Big O.** O(size of the input) time, O(longest line) memory: both files are streamed.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-5)

In [ ]:
# 18.5 Copy matching lines to another file
def copy_matching(source, target, needle):
    raise NotImplementedError

In [ ]:
def test_ex_18_5():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / 'log.txt').write_text('error: disk\nok\nerror: net')
        assert copy_matching(tmp / 'log.txt', tmp / 'out.txt', 'error') == 2
        assert (tmp / 'out.txt').read_text() == 'error: disk\nerror: net'
        assert copy_matching(tmp / 'log.txt', tmp / 'none.txt', 'panic') == 0
        assert (tmp / 'none.txt').read_text() == ''


test_ex_18_5()
print('18.5 passed')

### 18.6 · Averages from a CSV file  `Practice`

**Task.** A CSV file has the header `team,points` and one row per game. Write `average_by_team(path)` that returns a dictionary from team to its average points (a float). Read the rows with `csv.DictReader` (open the file with `newline=""`) and keep a total and a count per team.

**Example.** rows `red,10`, `blue,4`, `red,7` -> `{'red': 8.5, 'blue': 4.0}`

**Big O.** O(size of the file) time, O(T) space for T teams: keep a running total and count per team.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-18-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-18-6)

In [ ]:
# 18.6 Averages from a CSV file
def average_by_team(path):
    raise NotImplementedError

In [ ]:
def test_ex_18_6():
    with tempfile.TemporaryDirectory() as name:
        tmp = Path(name)
        (tmp / 'games.csv').write_text('team,points\nred,10\nblue,4\nred,7\n')
        (tmp / 'none.csv').write_text('team,points\n')
        assert average_by_team(tmp / 'games.csv') == {'red': 8.5, 'blue': 4.0}
        assert average_by_team(tmp / 'none.csv') == {}
        (tmp / 'quoted.csv').write_text('team,points\n"red, the best",3\n')
        assert average_by_team(tmp / 'quoted.csv') == {'red, the best': 3.0}


test_ex_18_6()
print('18.6 passed')